In [8]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Enable MPS fallback for operations not supported on MPS
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Set device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"Using device: {device}")

# Load model and tokenizer
print("Loading Qwen2.5-0.5B model and tokenizer...")
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# Use float16 instead of bfloat16 for Mac compatibility
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.float16,
    device_map="auto"  # This will use MPS if available
)


PyTorch version: 2.8.0.dev20250619
MPS available: True
Using device: mps
Loading Qwen2.5-0.5B model and tokenizer...


Device set to use mps


In [11]:
# Define chat template
def generate_response(user_input, system_prompt="You are a helpful AI assistant."):
    # Qwen2.5 uses ChatML format
    prompt = f"""
    <|im_start|>system
    {system_prompt}<|im_end|>
    <|im_start|>user
    {user_input}<|im_end|>
    <|im_start|>assistant\n"""
    
    print("\nGenerating response...")
    outputs = pipe(
        prompt,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
    )
    
    response = outputs[0]["generated_text"]
    # Extract just the assistant's response
    assistant_response = response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    return assistant_response

def chat_with_context(max_context=2):
    """
    Chat with the model while maintaining context of previous messages.
    
    Args:
        max_context (int): Maximum number of previous message pairs to keep in context
    """
    system_prompt = "You are a helpful AI assistant who provides concise and accurate information."
    conversation_history = []
    
    print("\n=== Qwen2.5-0.5B Chat with Context ===")
    print(f"System: {system_prompt}")
    print(f"Context: Keeping last {max_context} message pairs")
    print("Type 'exit' to quit")
    
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ["exit", "quit"]:
            break
        
        # Build the full conversation context
        full_prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        
        # Add conversation history (last max_context pairs)
        for user_msg, assistant_msg in conversation_history[-max_context:]:
            full_prompt += f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            full_prompt += f"<|im_start|>assistant\n{assistant_msg}<|im_end|>\n"
        
        # Add current user input
        full_prompt += f"<|im_start|>user\n{user_input}<|im_end|>\n"
        full_prompt += "<|im_start|>assistant\n"
        
        if user_input == "--context":
            print(f"\nFull prompt: {full_prompt}")
            continue

        print("\nGenerating response...")
        outputs = pipe(
            full_prompt,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
        )
        
        response = outputs[0]["generated_text"]
        # Extract just the assistant's response
        assistant_response = response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
        
        # Add to conversation history
        conversation_history.append((user_input, assistant_response))
        
        print(f"\nUser: {user_input}")
        print(f"\nQwen2.5: {assistant_response}")
        # print(f"\n[Context: {len(conversation_history)} message pairs stored]")

In [2]:

system_prompt = "You are a helpful AI assistant who provides concise and accurate information."
print("\n=== Qwen2.5-0.5B Chat ===")
print(f"System: {system_prompt}")
print("Type 'exit' to quit")

while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ["exit", "quit"]:
        break
        
    response = generate_response(user_input, system_prompt)
    print(f"\nQwen2.5: {response}") 


=== Qwen2.5-0.5B Chat ===
System: You are a helpful AI assistant who provides concise and accurate information.
Type 'exit' to quit


# Chat with context

In [12]:
chat_with_context(max_context=2)



=== Qwen2.5-0.5B Chat with Context ===
System: You are a helpful AI assistant who provides concise and accurate information.
Context: Keeping last 2 message pairs
Type 'exit' to quit

Generating response...

User: hey!

Qwen2.5: Hello! How can I assist you today?
